# FiGS Examples Notebook
This notebook shows you how to convert a capture into a GSplat and then simulate a drone trajectory within it. The trajectory is flown by an MPC expert and can be viewed through a video output from the onboard camera of the drone.

Some useful settings for interactive work

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

In [2]:
import os

try:
    preload = os.environ['LD_PRELOAD']
    print(preload)
except:
    print("cannot load LD_PRELOAD")
    pass
try:
    lib_path = os.environ['LD_LIBRARY_PATH']
    print(lib_path)
except:
    print("cannot load LD_LIBRARY_PATH")
    pass

cannot load LD_PRELOAD
/opt/intel/oneapi/tcm/1.5/lib:/opt/intel/oneapi/umf/1.1/lib:/opt/intel/oneapi/tcm/1.5/env/../lib:/opt/intel/oneapi/tbb/2023.1/env/../lib/intel64/gcc4.8:/opt/intel/oneapi/mkl/2026.1/lib:/opt/intel/oneapi/compiler/2026.1/opt/compiler/lib:/opt/intel/oneapi/compiler/2026.1/lib:/usr/local/cuda-13.3/lib64::/home/airlab/SousVide/FiGS/acados/lib


Importing the necessary libraries


In [3]:
import figs.render.capture_generation as pg
import figs.visualize.generate_videos as gv

from figs.simulator import Simulator
from figs.control.vehicle_rate_mpc import VehicleRateMPC
import logging
logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger("figs")
logger.setLevel(logging.INFO)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


#### 1. Define the capture (video name in gsplats/capture)

In [4]:
# # capture_name = "backroom"
# # capture_name = "button"
# capture_name = "area65_2026_03_27"
# capture_name = "area65_2026_04_02"
# capture_name = "aerial_arena_subset"
capture_name = "aerialarena_square_16x"

#### 2. Generate FiGS environment from capture

In [ ]:
# pg.generate_gsplat(scene_file_name=capture_name+"/images",use_images=True,capture_cfg_name="insta360x8")

pg.generate_gsplat(
    scene_file_name=capture_name,
    capture_cfg_name="rig_hloc",
    use_images=True,
)

INFO:figs:Processing flat camera rig with Nerfstudio rig bundle adjustment


[15:38:03] Using rig configuration from input:                             ]8;id=2326807;file:///home/airlab/SousVide/FiGS/nerfstudio/nerfstudio/process_data/colmap_converter_to_nerfstudio_dataset.py\colmap_converter_to_nerfstudio_dataset.py]8;;\:]8;id=2326808;file:///home/airlab/SousVide/FiGS/nerfstudio/nerfstudio/process_data/colmap_converter_to_nerfstudio_dataset.py#317\317]8;;\
           /home/airlab/SousVide/gsplats/capture/aerialarena_square_16x/ri                                              
           g_config.json                                                                                                

Copying rig images:   0%|          | 0/8472 [00:00<?, ?image/s]

Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000001.png already exists. Skipping copy.
Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000002.png already exists. Skipping copy.
Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000003.png already exists. Skipping copy.
Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000004.png already exists. Skipping copy.
Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000005.png already exists. Skipping copy.
Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000006.png already exists. Skipping copy.
Image /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images/NewGroup_Cam01/frame_000007.png already exists. Skipping copy.

[2026/08/05 15:38:04 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-r1024',
 'preprocessing': {'grayscale': True, 'resize_max': 1024}}


Found 8472 images in /home/airlab/SousVide/gsplats/workspace/aerialarena_square_16x/sfm/images.
Loaded SuperPoint model


 40%|███▉      | 3360/8472 [03:58<20:38,  4.13it/s]

#### 3. Define the scene and course

In [ ]:
#scene_name, course_name = "button", "button_prod"       # A short test trajectory where the drone approaches a button
# scene_name, course_name = "mid_gate", "traverse"        # Robustness trajecytory in the SOUS VIDE paper
scene_name, course_name = "backroom", "circuit"         # Cluttered trajectory in the SOUS VIDE paper
# scene_name, course_name = "images/splatfacto/2026-03-20_175241", "area65_line" 

#### 4. Simulate within the FiGS environment

In [ ]:
# Initialize the simulator and controller
sim = Simulator(scene_name,"eval_single","carl")
ctl = VehicleRateMPC("Viper",course_name,"carl")
# ctl = VehicleRateMPC("JesterButStronger",course_name,"carl")

# Use the ideal trajectory in VehicleRateMPC to get initial conditions and final time
t0,tf,x0 = ctl.tXUd[0,0],ctl.tXUd[-1,0],ctl.tXUd[0,1:11]

# Simulate the policy
Tro,Xro,Uro,Fro,Rgb,Dpt,Aux = sim.simulate(ctl,t0,tf,x0)

# Output the results as a video
gv.images_to_mp4(Rgb,course_name+'.mp4', ctl.hz)

Loading latest checkpoint from load_dir

✅ Done loading checkpoint from outputs/backroom/splatfacto/2025-01-16_122349/nerfstudio_models/step-000029999.ckpt

/home/airlab/SousVide/.venv/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
